In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 18
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 18
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [ ]:
!bash start_data_swarm.sh

In [ ]:
!bash stop_data_swarm.sh

In [ ]:
!python -m _tools.check_and_heal_data

In [ ]:
#полная проверка подготовленных на предыдущем этапе данных
!python -m _tools.verify_data

In [ ]:
#Запуск нескольких процессов в параллели
!bash start_swarm_amd.sh \
    --dataset_dir "data/processed/2000_2026_1d_18_3" \
    --bonus_ratio 0.2 \
    --min_delta 0.001 \
    --runs 500 --epochs 100 \
    --lr 2e-3 --l2_reg 1e-5 \
    --start_fold "fold_2010" \
    --factor 0.5 --patience 3 \
    --vram 10000/1 --stagger 0 \
    --keep 5 \
    --append \
    --arch attention
    #--arch cnn, conv1d+gru, mlp, attention
    #--track_trajectory
    #--init_pca_coord -148.0 -116.0 --init_pca_radius 20.0 \

In [ ]:
!./stop_swarm.sh
!python -m _tools.clean_lstm_models --keep 5

In [ ]:
!python run_all_ensembles.py --dataset_dir "data/processed/2000_2026_1d_18_3" --max_k 20

In [ ]:
%run _tools/plot_landscape.py --fold data/processed/2000_2026_1d_30_5/fold_2014

In [ ]:
%matplotlib inline
%run _tools/analyze_runs.py data/processed/2000_2026_1d_3_1/fold_2026 --runs 100 --arch attention

In [ ]:
!python -m _tools.prepare_rl_env

In [ ]:
!python -m _tools.check_env_data

In [11]:
!python -m _tools.find_healthy_checkpoints

🔍 Поиск лучших чекпоинтов Фазы 2 для каждого триала...
✅ Найдено: PPO_TradingEnv-... | Итер: 118 | Score: 1.52
✅ Найдено: PPO_TradingEnv-... | Итер: 110 | Score: 1.50
✅ Найдено: PPO_TradingEnv-... | Итер: 97 | Score: 1.29
✅ Найдено: PPO_TradingEnv-... | Итер: 120 | Score: 1.41
✅ Найдено: PPO_TradingEnv-... | Итер: 101 | Score: 1.92
✅ Найдено: PPO_TradingEnv-... | Итер: 98 | Score: 1.61
✅ Найдено: PPO_TradingEnv-... | Итер: 75 | Score: 1.73
✅ Найдено: PPO_TradingEnv-... | Итер: 99 | Score: 1.73

💾 Успешно сохранено 8 чекпоинтов в healthy_checkpoints.json


In [ ]:
!python -m _tools.train_rllib_pbt --population 8 --force --cpu --iterations 1400 --phase2_ratio 0.3 --phase3_ratio 0.7 --start_phase 3

In [ ]:
%load_ext tensorboard
%tensorboard --logdir data/processed/2000_2026_1d/rl_env/ray_results
#%tensorboard --logdir "C:\Users\Restorator\Documents\trader_test\trader_test\data\processed\2000_2026_1d\rl_env\ray_results"

In [10]:
!python -m _tools.export_to_git

🔍 Поиск лучшего чекпоинта через разбор OOS метаданных Ray Tune...
⚠️ Ошибка чтения /home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/ray_results/pbt_trading_bot/PPO_TradingEnv-v0_434a0_00002_2_2026-06-16_15-45-32/result.json: Expecting ',' delimiter: line 1 column 4760 (char 4759)
⚠️ Ошибка чтения /home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/ray_results/pbt_trading_bot/PPO_TradingEnv-v0_434a0_00004_4_2026-06-16_15-45-32/result.json: Expecting property name enclosed in double quotes: line 1 column 541 (char 540)
⚠️ Ошибка чтения /home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/ray_results/pbt_trading_bot/PPO_TradingEnv-v0_434a0_00005_5_2026-06-16_15-45-32/result.json: Expecting ',' delimiter: line 1 column 3000 (char 2999)
⚠️ Ошибка чтения /home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/ray_results/pbt_trading_bot/PPO_TradingEnv-v0_434a0_00003_3_2026-06-16_15-45-32/result.json: Expecting value: line 1 column 16534 (char 165

In [9]:
!python -m _tools.evaluate_agent --checkpoint "/home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/champions/best_model"

I0000 00:00:1781681965.402679  589712 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781681965.412958  589712 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781681974.463906  589712 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781681974.465850  589712 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
🔄 Восстановление агента из /home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/ray_results/pbt_trading_bot/PPO_TradingEnv-v0_434a0_00000_0_2026-06-16_15-45-32/checkpoint_000114/p